In [ ]:
import numpy as np
import sklearn
%matplotlib inline
import matplotlib as mpl 
import matplotlib.pyplot as plt

First of all the dataset is read and, from it, the feature (X) and label (y) arrays are created.

In [ ]:
import pandas as pd

df = pd.read_csv('HIGGS.csv.gz', header=None, nrows=500_000)

In [ ]:
y = df.iloc[:, 0].values
X = df.iloc[:, 1:].values        

The following commands are used to get a handle on the dataset, its structure and the values of its features.

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
X[0]

In [ ]:
y[0]

Then, the dataset is split into training, validation and testing groups. Considering also the labels, the next commands form 6 arrays. In this case, 20% of the full dataset is assigned to the testing, while 10% of the training set is kept for validation. 42 fixes the shuffling of the dataset so that it is reproducible. 

In [ ]:
from sklearn.model_selection import train_test_split

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.10, random_state=42)

In [ ]:
X_train.shape

In [ ]:
X_train[0]

The next command is used to visualise the different variables. In order to have a more complete view of their values, scatter plots are made for each feature, for the first 500 events, resulting in the plots visible below.

In [ ]:
features = [
    "lepton pt",
    "lepton eta",
    "lepton phi", 
    "missing energy magnitude", 
    "missing energy phi",
    "jet 1 pt", 
    "jet 1 eta", 
    "jet 1 phi",
    "jet 1 b-tag",
    "jet 2 pt",
    "jet 2 eta",
    "jet 2 phi",
    "jet 2 b-tag",
    "jet 3 pt",
    "jet 3 eta",
    "jet 3 phi",
    "jet 3 b-tag",
    "jet 4 pt",
    "jet 4 eta", 
    "jet 4 phi",
    "jet 4 b-tag",
    "m_jj",
    "m_jjj",
    "m_lv",
    "m_jlv",
    "m_bb",
    "m_wbb",
    "m_wwbb"
]

In [ ]:
import matplotlib.pyplot as plt

n_events = 500
X_sample = X_train[:n_events]  # first 500 events, all 28 features

fig, axes = plt.subplots(7, 4, figsize=(16, 20))  # 7 rows × 4 cols = 28 plots
axes = axes.flatten()  # makes it easier to index

for j, feature in enumerate(features):
    col_j = X_sample[:, j]  # all 500 values for feature j
    axes[j].scatter(range(n_events), col_j, s=2, color='steelblue', alpha=0.85)
    axes[j].set_title(feature, fontsize=11)
    axes[j].set_xlabel("Event number", fontsize=9)
    axes[j].set_ylabel("Value", fontsize=9)
    axes[j].grid(True, alpha=0.3)

plt.suptitle("Features' values (first 500 events)", fontsize=14, fontweight='bold', y=1)
plt.tight_layout()
plt.show()

The first Boosted Decision Tree model has just the basic parameters defined, those being "max_bins", and "max_iter". The first one determines the number of bins of the histograms which are built to discretise the features' values in order to make the training faster (255 is the default value). "max_iter", instead, is the maximum number of iterations of the boosting process, i.e. the maximum number of trees for binary classification. "validation_fraction" = 0.1 sets 10% of the training dataset for internal validation. 

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score

higgs_BDT_0 = HistGradientBoostingClassifier(max_bins=255, max_iter=100, validation_fraction=0.1) 

In [ ]:
%%time

higgs_BDT_0.fit(X_train, y_train)

In [ ]:
y_pred = higgs_BDT_0.predict(X_train)
train_acc = accuracy_score(y_train, y_pred)
print(f'train_acc: {train_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_0.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)
print(f"val_accuracy: {val_acc:.3f}")

In [ ]:
y_pred = higgs_BDT_0.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'test_acc: {test_acc:.3f}')

The accuracy shown by this first model can be considered good, given the simplicity of the algorithm. 

The first attempt to improve the Boosted Decision Tree results was made inserting and defining three other more parameters of the model: "max_depth", "learning_rate" and "l2_regularization". The depth of a tree is the number of edges to go from the root to the deepest leaf. So "max_depth" is the maximum depth of each tree. "learning_rate", also known as "shrinkage", is used as a multiplicative factor for the leaves values: "learning_rate" = 0.1 means that the prediction given by each tree is weighted only as the 10%. Finally, "l2_regularization" penalizes complexity to prevent overfitting by not allowing leaves to have large corrections if the data don't provide strong evidence for it. In the previous model (higgs_BDT_0) these three parameters had been maskedly set to their default values respectively of "None", "0.1" and "0", while in higgs_BDT_1 they have been kept the same as the default ones for "learning_rate" and "l2_regularization" and "max_depth" has been set to "3". In this way the only difference between the previous model and the following one is the different setting of the "max_depth" parameter. 

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier  
from sklearn.metrics import accuracy_score

higgs_BDT_1 = HistGradientBoostingClassifier(max_bins=255, max_iter=100, validation_fraction=0.1, max_depth=3, learning_rate=0.1, l2_regularization=0) 

In [ ]:
%%time

higgs_BDT_1.fit(X_train, y_train)

In [ ]:
y_pred = higgs_BDT_1.predict(X_train)
train_acc = accuracy_score(y_train, y_pred)
print(f'train_acc: {train_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_1.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)
print(f"val_accuracy: {val_acc:.3f}")

In [ ]:
y_pred = higgs_BDT_1.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'test_acc: {test_acc:.3f}')

The setting of a maximum value for the depth of the model seems to have worsened the performance of the algorithm, as it can be seen by the lower value of the accuracy. The fact that both training and test accuracies are lower than in the previous model seems to suggest that with no depth limit the trees would have developed further than just three, so imposing a maximum value worsened the structure of the classifier. 

For this reason, in the next model (higgs_BDT_2) the "max_depth" parameter has been set to 5, hoping to target the cause of the failing of the previous algorithm. 

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.metrics import accuracy_score

higgs_BDT_2 = HistGradientBoostingClassifier(max_bins=255, max_iter=100, validation_fraction=0.1, max_depth=5, learning_rate=0.1, l2_regularization=0)

In [ ]:
%%time

higgs_BDT_2.fit(X_train, y_train)

In [ ]:
y_pred = higgs_BDT_2.predict(X_train)
train_acc = accuracy_score(y_train, y_pred)
print(f'train_acc: {train_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_2.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)
print(f'val_acc: {val_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_2.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'test_acc: {test_acc:.3f}')

The increase in the value of the parameter describing the maximum depth reachable by a tree seems to have led to positive results, indicated by the improvements in the computed accuracies. 

With the goal of developing a more efficient algorithm, the next model follows the improvement obtained by higgs_BDT_2 and further increases "max_depth" to 7. 

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.metrics import accuracy_score

higgs_BDT_3 = HistGradientBoostingClassifier(max_bins=255, max_iter=100, validation_fraction=0.1, max_depth=7, learning_rate=0.1, l2_regularization=0)

In [ ]:
%%time

higgs_BDT_3.fit(X_train, y_train)

In [ ]:
y_pred = higgs_BDT_3.predict(X_train)
train_acc = accuracy_score(y_train, y_pred)
print(f'train_acc: {train_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_3.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)
print(f'val_acc: {val_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_3.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'test_acc: {test_acc:.3f}')

The increase in the "max_depth" parameter did indeed correspond to an increase in the calculated accuracies for this model. Further tests have been done to verify if a larger maximum depth of the trees would have further improved the accuracy, but the improvements were too minimal to justify a significant increase in this parameter value. "max_depth" = 7 has so been considered the best setting for this feature of the algorithm. 

The next models have so focused on other parameters of the tree, starting from the number of trees ("max_iter"), which has been set to 200 in the next algorithm higgs_BDT_4. 

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.metrics import accuracy_score

higgs_BDT_4 = HistGradientBoostingClassifier(max_bins=255, max_iter=200, validation_fraction=0.1, max_depth=7, learning_rate=0.1, l2_regularization=0)

In [ ]:
%%time

higgs_BDT_4.fit(X_train, y_train)

In [ ]:
y_pred = higgs_BDT_4.predict(X_train)
train_acc = accuracy_score(y_train, y_pred)
print(f'train_acc: {train_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_4.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)
print(f'val_acc: {val_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_4.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'test_acc: {test_acc:.3f}')

Increasing the value of the "max_iter" parameter led to a striking increase in all the computed accuracies, signaling the importance of the number of trees in the development of a boosted decision tree algorithm. In fact, this parameter describes the complexity of the model and so allows for a sharper distinction between signal and background. 

Given these results, another model was trained after having set "max_iter" to an even higher value: 300. 

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.metrics import accuracy_score

higgs_BDT_5 = HistGradientBoostingClassifier(max_bins=255, max_iter=300, validation_fraction=0.1, max_depth=7, learning_rate=0.1, l2_regularization=0)

In [ ]:
%%time

higgs_BDT_5.fit(X_train, y_train)

In [ ]:
y_pred = higgs_BDT_5.predict(X_train)
train_acc = accuracy_score(y_train, y_pred)
print(f'train_acc: {train_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_5.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)
print(f'val_acc: {val_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_5.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'test_acc: {test_acc:.3f}')

Similarly as the "max_depth" parameter, also for the "max_iter" case, a sort of "saturation" is reached, after which the improvements in the model's accuracies are much smaller than the parameter's increase. After having tried even larger values, "max_iter" = 300 seems the best option.   

The next step it to modify the learning rate parameter, which defines how much each tree contributes to the training of the algorithm. Usually the combination of a large number of trees and a small learning rate leads to a better model because each tree contributes only a small correction to the cumulative score, making the learning process more gradual and reducing the risk of any single tree's error dominating the final result. This needs to be compensated by a larger number of trees since each one contributes less. For this reason, the next model, higgs_BDT_6, was trained imposing "learning_rate" = 0.05. 

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.metrics import accuracy_score

higgs_BDT_6 = HistGradientBoostingClassifier(max_bins=255, max_iter=300, validation_fraction=0.1, max_depth=7, learning_rate=0.05, l2_regularization=0)

In [ ]:
%%time

higgs_BDT_6.fit(X_train, y_train)

In [ ]:
y_pred = higgs_BDT_6.predict(X_train)
train_acc = accuracy_score(y_train, y_pred)
print(f'train_acc: {train_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_6.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)
print(f'val_acc: {val_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_6.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'test_acc: {test_acc:.3f}')

Surprisingly, decreasing the learning rate led to a worse performance of the model. This could be caused by what was mentioned earlier: maybe a number of trees equal to 300 is not large enough to allow for a learning rate as small as 0.05. To test this hypothesis, first the number of trees was set to 1000, then the "n_iter_no_change" parameter was introduced and set, conventionally, to 10. This feature tells the algorithm to stop adding trees if there is no improvement in the validation accuracy after ten consecutive trees. In this way the model determines the optimal number of trees automatically, which will then be inspected via the "n_iter_" attribute. 

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.metrics import accuracy_score

higgs_BDT_7 = HistGradientBoostingClassifier(max_bins=255, max_iter=1000, validation_fraction=0.1, max_depth=7, learning_rate=0.05, l2_regularization=0, n_iter_no_change=10)

In [ ]:
%%time

higgs_BDT_7.fit(X_train, y_train)

In [ ]:
y_pred = higgs_BDT_7.predict(X_train)
train_acc = accuracy_score(y_train, y_pred)
print(f'train_acc: {train_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_7.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)
print(f'val_acc: {val_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_7.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'test_acc: {test_acc:.3f}')

In [ ]:
print(f"Trees used: {higgs_BDT_7.n_iter_}")

As expected, the required number of trees for a so small value of the learning rate was much higher than the previous 300, and by a large amount. This increase has in this way allowed to have an improvement in the accuracy, although not high as hoped.  

Given these poor results, which seem to not justify further increasing the number of trees, in the subsequent model, higgs_BDT_8, the "max_iter" parameter was set again to 300 and, for testing purposes, the learning rate was increased to 0.2. In this way checking if indeed the accuracy would have worsened as expected.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.metrics import accuracy_score

higgs_BDT_8 = HistGradientBoostingClassifier(max_bins=255, max_iter=300, validation_fraction=0.1, max_depth=7, learning_rate=0.2, l2_regularization=0)

In [ ]:
%%time

higgs_BDT_8.fit(X_train, y_train)

In [ ]:
y_pred = higgs_BDT_8.predict(X_train)
train_acc = accuracy_score(y_train, y_pred)
print(f'train_acc: {train_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_8.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)
print(f'val_acc: {val_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_8.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'test_acc: {test_acc:.3f}')

Surprisingly, the training accuracy actually increased with respect to higgs_BDT_5, the best model so far, with a lower learning rate than higgs_BDT_8. The cause of this behaviour is probably to be found in the reasoning made earlier about the number of trees. Given the fact that "max_iter" = 300 seems to be a small value for this parameter, probably this algorithm works better with a high learning rate of 0.2. For this reason, the next models will keep this last value for "learning_rate". 

The last parameter worth investigating in this analysis is "l2_regularization", which, as said before, has the goal of preventing complexity and, in particular, overfitting. Through all the last models this parameter has been set to zero. Now, considering also that the algorithm is becoming more complicated, it will be set to 0.1. Thus it will be verified if this parameter helps in improving the accuracy and avoiding overfitting, the degree of which can be estimated by looking at the difference between the training and validation accuracies: the largest the difference, the greater the risk of overfitting. 

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.metrics import accuracy_score

higgs_BDT_9 = HistGradientBoostingClassifier(max_bins=255, max_iter=300, validation_fraction=0.1, max_depth=7, learning_rate=0.2, l2_regularization=0.1)

In [ ]:
%%time

higgs_BDT_9.fit(X_train, y_train)

In [ ]:
y_pred = higgs_BDT_9.predict(X_train)
train_acc = accuracy_score(y_train, y_pred)
print(f'train_acc: {train_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_9.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)
print(f'val_acc: {val_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_9.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'test_acc: {test_acc:.3f}')

The introduction of the "l2_regularization" parameter has led to an increase of both the training and validation accuracies, although with different magnitudes. In particular, the training one continues to be larger, therefore resulting in an increment, although small, in the difference between these two parameters' values. The fact that the changing in value of the "l2_regularization" parameter didn't heavily affect the result, especially for the validation accuracy, suggests that probably the previous models weren't overfitting to begin with. 

To test the impact of this parameter, in the next, final, model, "l2_regularization" will be increased of a factor 10, setting it to 1. 

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.metrics import accuracy_score

higgs_BDT_10 = HistGradientBoostingClassifier(max_bins=255, max_iter=300, validation_fraction=0.1, max_depth=7, learning_rate=0.2, l2_regularization=1)

In [ ]:
%%time

higgs_BDT_10.fit(X_train, y_train)

In [ ]:
y_pred = higgs_BDT_10.predict(X_train)
train_acc = accuracy_score(y_train, y_pred)
print(f'train_acc: {train_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_10.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)
print(f'val_acc: {val_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_10.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'test_acc: {test_acc:.3f}')

Increasing "l2_regularization" to 1 didn't considerably change the accuracy. This is consistent with what was discussed earlier: since the model was not significantly overfitting, applying strong regularization was unnecessary and didn't affect the model's performance. 

In order to properly compare the eleven developed models, a more precise method can be found in plotting the ROC curves and computing the respective AUC values, as it has been done below. 

In [ ]:
from sklearn.metrics import roc_curve, auc

models = [
    higgs_BDT_0,
    higgs_BDT_1,
    higgs_BDT_2,
    higgs_BDT_3, 
    higgs_BDT_4,
    higgs_BDT_5,
    higgs_BDT_6,
    higgs_BDT_7,
    higgs_BDT_8,
    higgs_BDT_9,
    higgs_BDT_10
]

plt.figure(figsize=(8, 6))

for i, model in enumerate(models):
    y_scores_i = model.predict_proba(X_test) [:, 1]
    fpr_i, tpr_i, thresholds_i = roc_curve(y_test, y_scores_i)
    auc_value_i = auc(fpr_i, tpr_i)
    plt.plot(
        fpr_i, 
        tpr_i, 
        label=f'BDT_{i} (AUC = {auc_value_i:.3f})'
    )
    
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate (background efficiency)')
plt.ylabel('True Positive Rate (signal efficiency)')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Looking at the above plot and at the related AUC values, it is even more evident the performance drop observed passing from the first (higgs_BDT_0) to the second model (higgs_BDT_1) just by imposing a limit number for the trees, as it had already been observed from the accuracies. Furthermore, there are different models candidates for the one with the best performance, primarily higgs_BDT_7, higgs_BDT_9 and higgs_BDT_10. Recalling the characteristics of these Boosted Decision Trees, it should come as no surprise that higgs_BDT_7 has such a high AUC value, considering that this was the algorithm with a number of trees more than double the one of the others. Nevertheless, the results of this model, including the AUC value, don't justify the computational cost of such a high "max_iter". A similar logic can also be applied to the comparison between higgs_BDT_9 and higgs_BDT_10: as said before, changing the value of the regularization ("l2_regularization") didn't affect in a major way the models, therefore not justifying it being so high as 1. For these reasons, the Boosted Decision Tree that seems to have collected all the developments of the other models, without having ineffective complexity and bringing good performance results, appears to be higgs_BDT_9, characterised by "max_iter" = 300, "max_depth" = 7, "learning_rate" = 0.2 and "l2_regularization" = 0.1. 

Finally, for boosted decision trees it is possible to use the method "permutation_importance" to estimate the weight of every feature used in the model. It simply works by randomly shuffling that feature's values across all events and measuring how much the accuracy drops. If shuffling a feature causes a big accuracy drop, that feature was important. If accuracy barely changes, the feature wasn't contributing much. The result is then given in form of an histogram with the number of bins given by the number of features in the model and the importance on the y axis representing the percentage drop of the accuracy value for that specific feature's shuffling. This method was applied only to the best-performing model identified in the previous analysis (higgs_BDT_9). 

In [ ]:
from sklearn.inspection import permutation_importance

plt.figure(figsize=(8, 6))

result = permutation_importance(
    higgs_BDT_9,
    X_val,
    y_val,
    n_repeats=5,
    random_state=42
)

importances = result.importances_mean

plt.bar(
    features,
    importances,
    color='steelblue'
)

plt.title('BDT_9 feature importances')
plt.xlabel('Feature index')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


As it can be seen, the model seems to be highly dependent on the last features (from 22 to 28). These are high-level features, functions of the first 21, which are instead kinematic properties measured directly by the detector. The behaviour of the above plot is in line with the fact that the high-level features are purposely defined to better distinguish background and signal, so they play a larger role in the training of the models.  
Going into more detail, the three features with the largest impact ("m_bb", "m_wwbb", "m_wbb") are the ones describing the different combinations of invariant masses of one of the main decay processes of a Higgs boson, which involves both b quarks and W bosons. As described in Baldi et al. 2014 (https://arxiv.org/abs/1402.4735), the signal process takes place from the fusion of two gluons, leading to a heavy electrically-neutral Higgs boson (H0), which decays to a heavy electrically-charged Higgs boson (H±) and a W boson. H± bosons subsequently decay to a second W boson and the light Higgs boson, h0, which decays predominantly to a pair of bottom quarks.  
The other most important features ("m_jjj", "jet 1 pt", "m_jlv", "lepton pt", "missing energy magnitude") refer, instead, to the other primary decay process of the Higgs boson, where two oppositely charged W bosons are produced, which then decay one into two jets and the other into a lepton (l) and a neutrino (v). In particular, the "missing energy magnitude" describes exactly the energy of an object travelling in the transversal plane, that cannot be observed by the detector, associating it with the energy of the neutrino produced as final product in the process. 

To test the different impacts of the two types of observables, the next two models have been trained using a reduced set of features, in order to observe if the performances of the algorithms are indeed worse when considering just the low- (higgs_BDT_9_l) or the high-level (higgs_BDT_9_h) features. higgs_BDT_9, the model which achieved the best performances, has been used both as reference and as template.

In [ ]:
y_l = df.iloc[:, 0].values         
X_l = df.iloc[:, 1:22].values  

In [ ]:
from sklearn.model_selection import train_test_split

X_l_train_full, X_l_test, y_l_train_full, y_l_test = train_test_split(X_l, y_l, test_size=0.2, random_state=42) 

X_l_train, X_l_val, y_l_train, y_l_val = train_test_split(X_l_train_full, y_l_train_full, test_size=0.10, random_state=42) 

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.metrics import accuracy_score

higgs_BDT_9_l = HistGradientBoostingClassifier(max_bins=255, max_iter=300, validation_fraction=0.1, max_depth=7, learning_rate=0.2, l2_regularization=0.1)

In [ ]:
%%time

higgs_BDT_9_l.fit(X_l_train, y_l_train)

In [ ]:
y_pred = higgs_BDT_9_l.predict(X_l_train)
train_acc = accuracy_score(y_l_train, y_pred)
print(f'train_acc: {train_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_9_l.predict(X_l_val)
val_acc = accuracy_score(y_l_val, y_pred)
print(f'val_acc: {val_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_9_l.predict(X_l_test)
test_acc = accuracy_score(y_l_test, y_pred)
print(f'test_acc: {test_acc:.3f}')

In [ ]:
y_h = df.iloc[:, 0].values         
X_h = df.iloc[:, 22:].values  

In [ ]:
from sklearn.model_selection import train_test_split

X_h_train_full, X_h_test, y_h_train_full, y_h_test = train_test_split(X_h, y_h, test_size=0.2, random_state=42) 

X_h_train, X_h_val, y_h_train, y_h_val = train_test_split(X_h_train_full, y_h_train_full, test_size=0.10, random_state=42)

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.metrics import accuracy_score

higgs_BDT_9_h = HistGradientBoostingClassifier(max_bins=255, max_iter=300, validation_fraction=0.1, max_depth=7, learning_rate=0.2, l2_regularization=0.1)

In [ ]:
%%time

higgs_BDT_9_h.fit(X_h_train, y_h_train)

In [ ]:
y_pred = higgs_BDT_9_h.predict(X_h_train)
train_acc = accuracy_score(y_h_train, y_pred)
print(f'train_acc: {train_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_9_h.predict(X_h_val)
val_acc = accuracy_score(y_h_val, y_pred)
print(f'val_acc: {val_acc:.3f}')

In [ ]:
y_pred = higgs_BDT_9_h.predict(X_h_test)
test_acc = accuracy_score(y_h_test, y_pred)
print(f'test_acc: {test_acc:.3f}')

In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(8, 6))

y_scores = higgs_BDT_9.predict_proba(X_test) [:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
auc_value = auc(fpr, tpr)
plt.plot(
    fpr,
    tpr,
    label=f'BDT_9 (AUC = {auc_value:.3f})'
)

y_scores = higgs_BDT_9_l.predict_proba(X_l_test) [:, 1]
fpr, tpr, thresholds = roc_curve(y_l_test, y_scores)
auc_value = auc(fpr, tpr)
plt.plot(
    fpr,
    tpr,
    label=f'BDT_9_l (AUC = {auc_value:.3f})'
)

y_scores = higgs_BDT_9_h.predict_proba(X_h_test) [:, 1]
fpr, tpr, thresholds = roc_curve(y_h_test, y_scores)
auc_value = auc(fpr, tpr)
plt.plot(
    fpr,
    tpr,
    label=f'BDT_9_h (AUC = {auc_value:.3f})'
)

plt.plot([0, 1], [0, 1], 'k--', label='Random')

plt.xlabel('False Positive Rate (background efficiency)')
plt.ylabel('True Positive Rate (signal efficiency)')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

The above ROC curves, as well as the accuracy and AUC values, support the hypothesis: despite consisting of only seven features, the high-level feature set yields a model with better performance than the model trained exclusively on the 21 low-level features. Moreover, combining the low- and high-level features results in the best overall performance, indicating that the two feature sets provide complementary information.